# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardikkk-1209/ML_Pipeline/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# The written answer belongs in the markdown cell above.


## 1. My lane as an ML task

### Lane
**Refresh / Content Opportunity Scoring**

### Task type
**Classification with probability scoring for ranking.**

The core ML task is classification: predict whether a content page will experience a defined future performance decline. The model's predicted probability can then be used as a priority score to rank pages for human review. This matches the decision we framed earlier: which pages should a content or SEO team review first.

The model is not intended to decide automatically that a page must be refreshed.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# The final target will be constructed from the full warehouse,
# using a later observation window than the feature window.


## 2. Target or proxy

### Target

The final target will be an **observed future-decline label** built from the full daily warehouse.

For each content page, we will use a 30-day feature window ending at time **T** and compare it with the following 30-day outcome window. A page will be labelled **1 (future decline)** when its future 30-day clicks are at least 30% lower than its preceding 30-day clicks, subject to a minimum baseline-click threshold chosen before evaluation to avoid treating tiny-volume changes as meaningful. Otherwise it will be labelled **0**.

The label is therefore based on an outcome observed **after** the features, not on an existing FlyRank recommendation flag such as `is_declining`, `health_score`, or `needs_ctr_fix`.

The starter CSV in this notebook is useful for checking the available page-level signals, but the capstone target will be constructed from the full warehouse so that the outcome is genuinely future-looking.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# We will calculate precision@K on the held-out evaluation set after
# the future-decline label and model scores are available.


## 3. Success metric

The primary success metric will be **Precision@K**, where K is the top 10% of pages in the ranked review queue.

Precision@10% answers a practical question: **of the pages the model puts in the highest-priority 10% of the queue, what share actually experience the defined future decline?**

A good result is one that beats the fixed-rule baseline on the **same held-out evaluation set**. We will report the actual precision value rather than choosing a success threshold after seeing the results.

We will also report the decline base rate and recall at the same K so the ranking is interpreted in context.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd
from pathlib import Path

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

df.head()


In [5]:
# Confirm the page identifier exists and inspect candidate signals.
candidate_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "trend_direction",
    "trend_pct",
]
available = [c for c in candidate_cols if c in df.columns]

print("Available page-level columns:")
print(available)

if "content_id" in df.columns:
    print(f"Unique content pages: {df['content_id'].nunique():,}")
    print(f"Rows per content page (median): {df.groupby('content_id').size().median():.1f}")


### Unit of analysis

The unit of analysis for the capstone will be **one content page at one decision point in time**.

For the final feature table, each row will represent one `content_id` with features calculated from the 30-day window ending at time **T**. The future 30-day window after T supplies the observed target.

Therefore:

**1 row = 1 content page × 1 decision date**

This time-aware structure is important because it prevents information from the future outcome window from entering the features.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
# This cell records the baseline we will compare against.
# The baseline is intentionally simple and will be evaluated on the same
# held-out pages/time window as the ML model.
baseline_description = (
    "Rank pages by the largest negative recent-click trend, "
    "using only information available before the prediction point."
)
print(baseline_description)


## 5. Why ML beats a fixed rule here

A single rule such as “review pages whose recent clicks fell by more than 30%” is easy to understand, so it will be our baseline rather than something we dismiss.

ML earns its place only if it improves the ranking using several signals together, such as recent impressions and clicks, CTR, average position, engagement, content age, query-level characteristics, and recent volatility. These signals can interact in ways that are difficult to capture with one manually chosen threshold.

We will therefore compare the ML probability ranking against the fixed-rule baseline on the **same held-out evaluation set**. If ML does not improve Precision@10%, we will report that honestly and prefer the simpler baseline.

The final output remains **decision support for a human reviewer**, not an automatic refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.